In [4]:
import os
from pykeen.hpo import hpo_pipeline
from pykeen.triples import TriplesFactory

c:\Users\zalak\miniconda3\envs\dicee_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# Load Dataset
dataset_dir = r"C:\my-code\Robust Embedding\dicee\uml_dataset"


# Create PyKEEN Triples Factories for each split
training_factory = TriplesFactory.from_path(
    os.path.join(dataset_dir, "train.txt"),
    create_inverse_triples=True # Creates reciprocal links
)

# Validation and Test factories must share the entity-to-id mapping from Training
validation_factory = TriplesFactory.from_path(
    os.path.join(dataset_dir, "valid.txt"),
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

test_factory = TriplesFactory.from_path(
    os.path.join(dataset_dir, "test.txt"),
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

In [13]:
# Set optimization pipeline

hpo_result = hpo_pipeline(

    # Pass data factories
    training=training_factory,
    validation=validation_factory,
    testing=test_factory,

    # Target Model
    model='ComplEx',

    # Define Hyperparameter Range
    model_kwargs_ranges=dict(

        embedding_dim=dict( type=int, low=64, high=256, step=64 )),

        optimizer_kwargs_ranges=dict( lr=dict(type=float, low=0.001, high=0.05, scale='log') ),

        # Training configurations
        training_kwargs=dict( num_epochs=150, batch_size=256 ),

        # HPO Loop Settings
        n_trials=5,
        metric='mrr',
        device='cpu'
)

print("Optimization Loop Completed")
# Get Result
print(f"Best Parameters Found: {hpo_result.best_kvs}")

[I 2026-06-21 18:51:55,052] A new study created in memory with name: no-name-bab32c7d-a770-442b-840d-b1c6718de9d8
No random seed is specified. Setting to 1173307281.
INFO:pykeen.triples.triples_factory:Creating inverse triples.
c:\Users\zalak\miniconda3\envs\dicee_env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training epochs on cpu: 100%|██████████| 150/150 [01:16<00:00,  1.97epoch/s, loss=0.247, prev_loss=0.245]
Evaluating on cpu:   0%|          | 0.00/652 [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 652/652 [00:00<00:00, 4.06ktriple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 0.20s seconds
[I 2

Optimization Loop Completed


AttributeError: 'HpoPipelineResult' object has no attribute 'best_kvs'